<a href="https://colab.research.google.com/github/kolesnikconstantine-a11y/studymate-ai-assistant/blob/main/Final_%D0%9A%D0%BE%D0%BB%D1%96%D1%81%D0%BD%D0%B8%D0%BA_%D0%9A%D0%BE%D1%81%D1%82%D1%8F%D0%BD%D1%82%D0%B8%D0%BD_%D0%90%D0%BD%D0%B0%D1%82%D0%BE%D0%BB%D1%96%D0%B9%D0%BE%D0%B2%D0%B8%D1%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StudyMate — освітній асистент
**Курс:** AI Fundamentals · Product Management


## 0. Налаштування

In [ ]:
!pip install -q \
    "langchain==1.3.11" \
    "langchain-openai==1.3.3" \
    "langchain-community==0.4.2" \
    "langgraph==1.2.7" \
    "langchainhub==0.1.21" \
    "python-dotenv==1.2.2" \
    "wikipedia==1.4.0" \
    "numexpr==2.14.1" \
    "langchain-google-genai" \
    "numpy<3" 2>&1 | grep -v "dependency conflicts"

In [ ]:
import os
from datetime import datetime
from typing import List, Dict, Any
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

print("✅ Імпорти виконано.")

✅ Імпорти виконано.


In [ ]:
import getpass
import os


def _load_key(name: str, prompt: str, required: bool = False) -> str | None:
    val = None
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        val = None

    if not val and required:
        val = getpass.getpass(prompt)

    if required and not val:
        raise ValueError(f"{name} не налаштовано. Додай ключ у Colab Secrets або введи вручну.")

    if val:
        os.environ[name] = val
    return val


_load_key("OPENAI_API_KEY", "Введіть OpenAI API key: ", required=False)
_load_key("GOOGLE_API_KEY", "Введіть Google Gemini API key: ", required=False)

if not os.environ.get("OPENAI_API_KEY") and not os.environ.get("GOOGLE_API_KEY"):
    raise ValueError("Потрібен хоча б один ключ: OPENAI_API_KEY або GOOGLE_API_KEY.")

print("✅ Ключі провайдерів перевірено (значення не виводяться)")

✅ Ключі провайдерів перевірено (значення не виводяться)


## 1. Бази даних

In [ ]:
import json
import urllib.request
from pathlib import Path

URLS = {
    "periodic_table.json": "https://drive.google.com/uc?export=download&id=1lncuL7o7v4elD_navjJZQBt5jhNXQh2B",
    "formulas_db.json": "https://drive.google.com/uc?export=download&id=1jihMimeVR8pk_YZs7enjECHfjKk0fez_",
}


def load_json(name: str, fallback):
    local_file = Path(name)
    if local_file.exists():
        return json.loads(local_file.read_text(encoding="utf-8"))

    url = URLS.get(name, "").strip()
    if url:
        urllib.request.urlretrieve(url, local_file)
        return json.loads(local_file.read_text(encoding="utf-8"))

    return fallback


PERIODIC_TABLE = load_json("periodic_table.json", {})
FORMULAS_DB = load_json("formulas_db.json", {})

print(f"✅ Завантажено елементів: {len(PERIODIC_TABLE)}")
print(f"✅ Завантажено категорій формул: {len(FORMULAS_DB)}")

✅ Завантажено елементів: 118
✅ Завантажено категорій формул: 3


In [ ]:
def convert_physics(value, from_unit, to_unit):
    key = (from_unit.strip(), to_unit.strip())
    conversions = {
        # Швидкість
        ("км/год", "м/с"): value / 3.6,
        ("м/с", "км/год"): value * 3.6,
        # Температура
        ("C", "F"): value * 9 / 5 + 32,
        ("F", "C"): (value - 32) * 5 / 9,
        ("C", "K"): value + 273.15,
        ("K", "C"): value - 273.15,
        # Енергія
        ("Дж", "кал"): value / 4.184,
        ("кал", "Дж"): value * 4.184,
        ("кДж", "Дж"): value * 1000,
        ("Дж", "кДж"): value / 1000,
        # Тиск
        ("атм", "Па"): value * 101325,
        ("Па", "атм"): value / 101325,
        ("кПа", "Па"): value * 1000,
        ("Па", "кПа"): value / 1000,
        # Маса (для переведення в СІ — кг)
        ("г", "кг"): value / 1000,
        ("кг", "г"): value * 1000,
        ("т", "кг"): value * 1000,
        ("кг", "т"): value / 1000,
        # Відстань / довжина (для переведення в СІ — м)
        ("см", "м"): value / 100,
        ("м", "см"): value * 100,
        ("мм", "м"): value / 1000,
        ("м", "мм"): value * 1000,
        ("км", "м"): value * 1000,
        ("м", "км"): value / 1000,
        # Час (для переведення в СІ — с)
        ("хв", "с"): value * 60,
        ("с", "хв"): value / 60,
        ("год", "с"): value * 3600,
        ("с", "год"): value / 3600,
    }
    return conversions.get(key)


total_formulas = sum(len(f) for f in FORMULAS_DB.values())
by_subject = ", ".join(f"{subj}: {len(items)}" for subj, items in FORMULAS_DB.items())
print(f"✅ База формул: {total_formulas} ({by_subject})")
print(f"✅ База елементів: {len(PERIODIC_TABLE)}")

✅ База формул: 500 (математика: 173, фізика: 175, хімія: 152)
✅ База елементів: 118


## 2. Інструменти (@tool)

In [ ]:
@tool
def formula_lookup(query: str) -> str:
    """Шукає формули з математики, фізики, хімії.
    Використовуй коли учень питає формулу, як обчислити, пояснити формулу.
    Приклад: 'кінетична енергія', 'теорема піфагора', 'сила струму'"""
    if not query.strip():
        return "Вкажи назву формули або тему."

    stop_words = {
        "формула", "формули", "формулу", "закон", "закони",
        "рівняння", "теорема", "правило", "вираз", "величина",
        "знайти", "обчислити", "розрахувати", "порахувати",
        "для", "при", "через", "або", "про", "яка", "який", "яке",
    }

    q = query.lower().strip()
    raw_tokens = [w for w in re.findall(r"[a-zа-яіїєґ0-9_]+", q) if len(w) > 1]
    tokens = [w for w in raw_tokens if w not in stop_words] or raw_tokens
    stems = [w[:6] if len(w) > 6 else w for w in tokens]

    def _format_formula(subject: str, name: str, data: dict) -> str:
        res = [
            f"{name.upper()} ({subject})",
            f"Формула: {data['формула']}",
            "Змінні:",
        ]
        for var, meaning in data.get("змінні", {}).items():
            res.append(f"  {var} — {meaning}")
        if "приклад" in data:
            res.append(f"\nПриклад: {data['приклад']}")
        if "примітка" in data:
            res.append(f"\nПримітка: {data['примітка']}")
        return "\n".join(res)

    def _is_template(data: dict) -> bool:
        note = data.get("примітка", "").lower()
        return "навчальна картка" in note or "окремий випадок" in note

    # 1. Точний або прямий збіг фрази у назві чи описі змінних
    exact_candidates = []
    for subject, formulas in FORMULAS_DB.items():
        for name, data in formulas.items():
            nl = name.lower()
            vars_meanings = " ".join(v.lower() for v in data.get("змінні", {}).values())
            penalty = -2 if _is_template(data) else 0

            if q == nl:
                exact_candidates.append((100 + penalty, subject, name, data))
            elif q in nl:
                exact_candidates.append((80 + penalty, subject, name, data))
            elif len(q) >= 5 and q in vars_meanings:
                exact_candidates.append((65 + penalty, subject, name, data))

    if exact_candidates:
        exact_candidates.sort(key=lambda x: (-x[0], len(x[2])))
        _, best_subj, best_name, best_data = exact_candidates[0]
        return _format_formula(best_subj, best_name, best_data)

    # 2. Ранжування за змістовними токенами та основами слів
    scored = []
    for subject, formulas in FORMULAS_DB.items():
        for name, data in formulas.items():
            nl = name.lower()
            vars_text = " ".join(
                f"{k} {v}".lower() for k, v in data.get("змінні", {}).items()
            )
            example_text = data.get("приклад", "").lower()

            score = 0
            matched_stems = 0
            for stem in stems:
                in_name = stem in nl
                in_vars = stem in vars_text
                in_ex = stem in example_text
                if in_name or in_vars or in_ex:
                    matched_stems += 1
                if in_name:
                    score += 4
                if in_vars:
                    score += 2
                if in_ex:
                    score += 1

            if stems and matched_stems == len(stems):
                score += 5
            if _is_template(data):
                score -= 3

            if score >= 3:
                scored.append((score, matched_stems, name, subject, data))

    if scored:
        scored.sort(key=lambda x: (-x[0], -x[1], len(x[2])))
        if len(scored) == 1 or scored[0][0] - scored[1][0] >= 3:
            _, _, best_name, best_subj, best_data = scored[0]
            return _format_formula(best_subj, best_name, best_data)

        top_matches = [f"  • {name} ({subj})" for _, _, name, subj, _ in scored[:7]]
        return "Знайдено кілька близьких формул. Уточни, яка саме потрібна:\n" + "\n".join(top_matches)

    return f"Формулу '{query}' не знайдено. Спробуй уточнити запит (математика, фізика, хімія)."

@tool
def unit_converter_physics(query: str) -> str:
    """Конвертує фізичні одиниці: швидкість, температуру, енергію, тиск.
    Приклад: '100 км/год у м/с', '37 цельсій у фаренгейт'"""
    if not query.strip():
        return "Вкажи значення та одиниці. Наприклад: '100 км/год у м/с'."
    numbers = re.findall(r"-?\d+(?:[.,]\d+)?", query)
    if not numbers:
        return "Не знайдено числа."
    value = float(numbers[0].replace(",", "."))
    q = query.lower()
    unit_aliases = {
        "км/год": ["км/год", "км/г", "km/h"],
        "м/с": ["м/с", "m/s"],
        "C": ["цельсій", "°c", "celsius"],
        "F": ["фаренгейт", "°f", "fahrenheit"],
        "K": ["кельвін", " k ", "kelvin"],
        "кДж": ["кілоджоуль", "кдж", "kj"],
        "Дж": ["джоуль", "дж"],
        "кал": ["калорія", "кал", "cal"],
        "атм": ["атмосфера", "атм"],
        "кПа": ["кілопаскаль", "кпа", "kpa"],
        "Па": ["паскаль", "па", "pascal"],
        "кг": ["кілограм", "кг", "kg"],
        "г": ["грам", " г", "g"],
        "т": ["тонна", "тонн", " т"],
        "км": ["кілометр", "км", "km"],
        "см": ["сантиметр", "см", "cm"],
        "мм": ["міліметр", "мм", "mm"],
        "м": ["метр", " м", "m"],
        "год": ["година", "годин", "год", "h"],
        "хв": ["хвилина", "хвилин", "хв", "min"],
        "с": ["секунда", "секунд", " с", "sec"],
    }
    def detect(text):
        for canonical, aliases in unit_aliases.items():
            if any(a in text for a in aliases): return canonical
        return None
    from_unit = detect(q)
    parts = re.split(r"\s+[уводвдо]+\s+", q)
    to_unit = detect(parts[-1]) if len(parts) > 1 else None
    if not from_unit or not to_unit:
        return "Не розпізнано одиниці. Доступні: км/год↔м/с, C↔F↔K, Дж↔кал, атм↔Па"
    result = convert_physics(value, from_unit, to_unit)
    if result is None:
        return f"Конвертація {from_unit} → {to_unit} не підтримується."
    return f"{value} {from_unit} = {result:.4g} {to_unit}"

def _format_element(symbol, data):
    return (f"{data['name_ua']} ({data['name_en']}) — {symbol}\n\n"
            f"Атомний номер: {data['number']}\nМаса: {data['mass']} г/моль\n"
            f"Група: {data['group']}\nКонфігурація: {data['electron_config']}\n\n"
            f"Властивості: {data['properties']}\nЗастосування: {data['uses']}")

@tool
def periodic_table(query: str) -> str:
    """Інформація про хімічний елемент: маса, конфігурація, властивості.
    Приклад: 'Fe', 'залізо', 'золото'"""
    if not query.strip():
        return "Вкажи символ або назву елемента."
    q = query.lower().strip()
    for symbol, data in PERIODIC_TABLE.items():
        if q == symbol.lower(): return _format_element(symbol, data)
    for symbol, data in PERIODIC_TABLE.items():
        if q in data["name_ua"].lower() or data["name_ua"].lower() in q or q in data["name_en"].lower():
            return _format_element(symbol, data)
    available = ", ".join(f"{s} ({d['name_ua']})" for s, d in PERIODIC_TABLE.items())
    return f"Елемент '{query}' не знайдено. Доступні: {available}."

@tool
def study_planner(query: str) -> str:
    """Планує підготовку до іспиту: теми/день, чи вистачить часу.
    Приклад: '15 тем, 10 днів, 3 години на день'"""
    if not query.strip():
        return "Вкажи: кількість тем, днів, годин на день."
    numbers = re.findall(r"\d+(?:[.,]\d+)?", query)
    if len(numbers) < 3:
        return "Потрібно 3 числа: теми, дні, години. Приклад: '15 тем, 10 днів, 3 години'."
    topics = int(numbers[0]); days = int(numbers[1])
    hours = float(numbers[2].replace(",", "."))
    if topics <= 0 or days <= 0 or hours <= 0: return "Всі значення > 0."
    if hours > 16: return f"{hours} годин/день — нереалістично."
    q = query.lower()
    h_per_topic = 2.5 if "важк" in q else 1.0 if "легк" in q else 1.5
    difficulty = "важка" if "важк" in q else "легка" if "легк" in q else "середня"
    needed = topics * h_per_topic; available = days * hours
    result = [f"ПЛАН ПІДГОТОВКИ\n", f"Тем: {topics}, Днів: {days}, Годин/день: {hours}",
              f"Складність: {difficulty}\n",
              f"Потрібно: {needed:.1f} год | Доступно: {available:.1f} год"]
    if available >= needed:
        result.append(f"✅ Часу достатньо! Резерв: {available-needed:.1f} год")
    else:
        result.append(f"⚠️ Дефіцит: {needed-available:.1f} год")
    result += [f"\nТем/день: {topics/days:.1f}",
               f"Помодоро (25хв): {int(hours*60/25)} сесій/день",
               "\nПоради:", "1. Складне — зранку", "2. Активне повторення",
               "3. Останній день — повторення", "4. Сон ≥7 годин"]
    return "\n".join(result)

tools = [formula_lookup, unit_converter_physics, periodic_table, study_planner]
print("✅ Інструменти StudyMate створено.")


✅ Інструменти StudyMate створено.


## 3. Модель

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

# Прапор для презентації:
# False = штатний режим (основна модель gemini-4-hydrogen)
# True  = симуляція збою основної моделі для демонстрації fallback на gpt-6-luna
SIMULATE_PRIMARY_FAILURE = False


def build_candidate_models(simulate_failure: bool) -> list:
    """Формує ланцюжок моделей (Google Gemini -> OpenAI) у єдиному місці."""
    models = []

    if os.environ.get("GOOGLE_API_KEY"):
        models.append(
            ChatGoogleGenerativeAI(
                model="gemini-nonexistent" if simulate_failure else "gemini-4-hydrogen",
                max_output_tokens=1600,
            )
        )

    if os.environ.get("OPENAI_API_KEY"):
        models.append(
            ChatOpenAI(
                model="gpt-6-luna",
                reasoning={"effort": "none"},
                max_tokens=1600,
                timeout=30,
                use_responses_api=True,
            )
        )

    if not models:
        raise RuntimeError("Не вдалося створити жодної моделі — перевір API-ключі.")

    return models


candidate_models = build_candidate_models(SIMULATE_PRIMARY_FAILURE)
chat_model = candidate_models[0]

sim_status = (
    "ВКЛЮЧЕНО (імітація збою основної моделі)"
    if SIMULATE_PRIMARY_FAILURE
    else "ВИМКНЕНО (штатний режим)"
)
print(f"✅ Моделі налаштовано ({len(candidate_models)}).")
print(f"🎛️ Режим симуляції fallback: {sim_status}")
print(f"   Пріоритет 1: {getattr(candidate_models[0], 'model', 'unknown')}")
if len(candidate_models) > 1:
    print(f"   Резерв (fallback): {getattr(candidate_models[1], 'model', 'unknown')}")

✅ Моделі налаштовано (2).
🎛️ Режим симуляції fallback: ВИМКНЕНО (штатний режим)
   Пріоритет 1: gemini-4-hydrogen
   Резерв (fallback): gpt-6-luna


## 4. Агент

In [ ]:
SYSTEM_PROMPT = r"""
Ти — освітній асистент StudyMate, помічник для учнів і студентів точних наук.

Твої завдання:
- знаходити формули з математики, фізики та хімії і пояснювати їх;
- конвертувати фізичні одиниці вимірювання;
- надавати інформацію про хімічні елементи;
- допомагати планувати підготовку до іспитів;
- розпізнавати та розбирати умови задач за фото: виділяти ключові дані («Дано» / «Знайти»),
 показувати кроки конвертації одиниць та підбирати потрібну формулу;
- відповідати українською мовою, чітко і зрозуміло.

Правила використання інструментів:
- Коли питають формулу або як щось обчислити — використовуй formula_lookup.
- Коли потрібна конвертація фізичних величин — використовуй unit_converter_physics.
- Коли питають про хімічний елемент — використовуй periodic_table.
- Коли потрібно скласти план підготовки до іспиту — використовуй study_planner.
- Якщо учень надсилає фото або текст задачі (особливо з фізики):
  1) виділи ключові дані («Дано» та «Знайти»);
  2) покажи кроки конвертації одиниць у систему СІ за допомогою unit_converter_physics;
  3) підбери формулу через formula_lookup та поясни змінні.

Важливо:
- Пояснюй, а не просто відповідай. Після формули завжди поясни, що означають змінні.
- Якщо учень просить "пояснити ще раз" або "простіше" — перефразуй відповідь іншими словами.
- Не вигадуй формули самостійно — використовуй лише інструмент formula_lookup.
- Якщо формули немає в базі — чесно скажи про це і запропонуй уточнити запит.
- Не видавай себе за джерело абсолютної істини: якщо не впевнений — скажи про це.
- Якщо запит виходить за межі точних наук, ввічливо поясни свою спеціалізацію.
- У текстовому виводі (без рендеру LaTeX) пиши формули просто і читабельно — наприклад,
 E_k = (m * v²) / 2 або E_k = mv² / 2, без складних LaTeX-команд типу \[ \], \frac чи \text{...}.
- Якщо учень ввів лише прізвище вченого чи загальне слово (наприклад, «ньютон»),
 спершу наведи знайдену формулу з бази (якщо вона є) і запитай, чи мався на увазі
  саме цей закон/формула, чи інше поняття.
"""

agents = [
    create_agent(model=m, tools=tools, system_prompt=SYSTEM_PROMPT)
    for m in candidate_models
]
agent = agents[0]

def set_demo_fallback(simulate_failure: bool):
    """Перемикає режим демо під час презентації та перебудовує ланцюжок агентів."""
    global SIMULATE_PRIMARY_FAILURE, candidate_models, agents, agent, chat_model

    SIMULATE_PRIMARY_FAILURE = simulate_failure
    candidate_models = build_candidate_models(simulate_failure)
    agents = [
        create_agent(model=m, tools=tools, system_prompt=SYSTEM_PROMPT)
        for m in candidate_models
    ]
    agent = agents[0]
    chat_model = candidate_models[0]

    status = (
        "ВКЛЮЧЕНО (імітація збою основної моделі)"
        if simulate_failure
        else "ВИМКНЕНО (працює основна gemini-3.8-flash)"
    )
    print(f"🎛️ Режим симуляції fallback: {status}")


def _get_model_label(model_obj, idx: int) -> str:
    return (
        getattr(model_obj, "model", None)
        or getattr(model_obj, "model_name", None)
        or f"Model #{idx}"
    )

import textwrap

def wrap_text(text: str, width: int = 92) -> str:
    """Переносить довгі рядки по ширині, зберігаючи абзаци та списки."""
    lines = []
    for line in str(text).splitlines():
        if not line.strip():
            lines.append("")
            continue
        stripped = line.lstrip()
        indent = line[: len(line) - len(stripped)]
        if stripped.startswith(("- ", "* ", "• ")):
            marker = stripped[:2]
            body = stripped[2:]
            lines.append(
                textwrap.fill(
                    body,
                    width=width,
                    initial_indent=f"{indent}{marker}",
                    subsequent_indent=f"{indent}{' ' * len(marker)}",
                    break_long_words=False,
                )
            )
        else:
            lines.append(
                textwrap.fill(
                    line,
                    width=width,
                    initial_indent=indent,
                    subsequent_indent=indent,
                    break_long_words=False,
                )
            )
    return "\n".join(lines)


def _extract_last_text(result: dict) -> str:
    out_messages = result.get("messages", []) if isinstance(result, dict) else []
    if not out_messages:
        return ""
    last_msg = out_messages[-1]
    raw = last_msg.content if hasattr(last_msg, "content") else (
        last_msg.get("content", "") if isinstance(last_msg, dict) else ""
    )
    # У деяких моделей (зокрема Gemini) content може бути списком блоків
    if isinstance(raw, list):
        parts = []
        for item in raw:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and item.get("text"):
                parts.append(item["text"])
        return "\n".join(parts).strip()
    return str(raw or "").strip()


def invoke_with_fallback(messages: list) -> dict:
    """Послідовно викликає агентів (OpenAI / Google Gemini), поки не отримає непусту відповідь."""
    last_error = None

    for idx, current_agent in enumerate(agents, start=1):
        label = _get_model_label(candidate_models[idx - 1], idx)
        try:
            result = current_agent.invoke({"messages": messages})
            text = _extract_last_text(result)
            if text:
                if idx > 1:
                    print(f"ℹ️ Відповідь отримано через fallback-модель: {label}")
                return result
            print(f"⚠️ {label} повернула порожню відповідь. Пробую наступну модель...")
        except Exception as e:
            last_error = e
            print(f"⚠️ Помилка у {label} ({type(e).__name__}): {e}. Перемикаюсь на наступну...")

    raise RuntimeError(f"Усі моделі (OpenAI / Google) не змогли відповісти. Остання помилка: {last_error}")


print("✅ Агент StudyMate з fallback OpenAI ↔ Google готовий.")


✅ Агент StudyMate з fallback OpenAI ↔ Google готовий.


## 5. Інтерактивний чат

In [ ]:
import base64
from google.colab import files
from langchain_core.messages import HumanMessage


def upload_task_image() -> HumanMessage | None:
    """Завантажує фото в Colab і формує мультимодальний HumanMessage для LangChain."""
    print("📎 Оберіть зображення (JPG/PNG)...")
    uploaded = files.upload()
    if not uploaded:
        return None

    filename = next(iter(uploaded))
    image_bytes = uploaded[filename]
    mime_type = "image/png" if filename.lower().endswith(".png") else "image/jpeg"
    b64_image = base64.b64encode(image_bytes).decode("utf-8")

    return HumanMessage(
        content=[
            {
                "type": "text",
                "text": (
                    "Уважно прочитай і перепиши все, що зображено на цій картинці (текст, числа, формули).\n"
                    "Якщо на зображенні є умова задачі або величини для переведення — додатково:\n"
                    "1) випиши «Дано» та «Знайти»;\n"
                    "2) переведи нестандартні одиниці в систему СІ через unit_converter_physics;\n"
                    "3) підбери формулу через formula_lookup і поясни кроки розв'язання."
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{mime_type};base64,{b64_image}",
                    "detail": "high",
                },
            },
        ]
    )

In [ ]:
import base64
from google.colab import files


def run_studymate_session():
    print("\n" + "=" * 60)
    print("📚 StudyMate — освітній AI-асистент")
    print("=" * 60)
    print("exit — завершити, /reset — скинути контекст, /photo — завантажити фото задачі")
    print("-" * 60)
    messages = []

    while True:
        try:
            user_input = input("\n👤 Ви: ").strip()
            if not user_input:
                continue

            if user_input.lower() in ("exit", "вихід", "/exit", "quit"):
                print("👋 До побачення!")
                break

            if user_input.lower() in ("/reset", "reset"):
                messages = []
                print("🔄 Контекст скинуто.")
                continue

            if user_input.lower() in ("/photo", "фото"):
                photo_msg = upload_task_image()
                if not photo_msg:
                    print("⚠️ Файл не обрано.")
                    continue
                messages.append(photo_msg)
                print("🤔 Аналізую фото задачі...")

                result = invoke_with_fallback(messages)
                messages = result["messages"]
                last = messages[-1]
                ans = _extract_last_text(result)
                print(f"✅ {wrap_text(ans, width=92)}")
                continue

            messages.append({"role": "user", "content": user_input})
            print("🤔 Думаю...")

            result = invoke_with_fallback(messages)
            messages = result["messages"]
            last = messages[-1]
            ans = _extract_last_text(result)
            print(f"✅ {wrap_text(ans, width=92)}")


        except KeyboardInterrupt:
            print("\n👋 Перервано.")
            break
        except Exception as e:
            print(f"❌ Помилка: {e}")


run_studymate_session()


📚 StudyMate — освітній AI-асистент
exit — завершити, /reset — скинути контекст, /photo — завантажити фото задачі
------------------------------------------------------------

👤 Ви: exit
👋 До побачення!


## 6. Автоматичне тестування

In [ ]:
def run_automatic_tests():
    test_queries = [
        # 1. Звичайне запитання без tool
        "Привіт! Що ти вмієш?",
        # 2. Пошук формули (formula_lookup)
        "Яка формула кінетичної енергії?",
        # 3. Продовження контексту — прохання пояснити простіше
        "Поясни цю формулу простішими словами.",
        # 4. Конвертація одиниць (unit_converter_physics)
        "Переведи 100 км/год у м/с",
        # 5. Інформація про елемент (periodic_table)
        "Розкажи про елемент Fe",
        # 6. Планування підготовки (study_planner)
        "Допоможи спланувати підготовку: 15 тем, 10 днів, 3 години на день",
        # 7. Перевірка пам'яті — посилання на попередній контекст
        "А чи вистачить часу, якщо у мене лише 7 днів?",
        # 8. Формула, якої немає в базі — перевірка чесності бота
        "Яка формула квантової заплутаності?",
       # 9. Запит поза темою — перевірка меж поведінки
        "Порекомендуй фільм на вечір",
    ]
    messages = []
    print("\n" + "="*60 + "\n🧪 ТЕСТУВАННЯ StudyMate\n" + "="*60)
    for i, q in enumerate(test_queries, 1):
        print(f"\nТест #{i}\n" + "-"*60 + f"\n👤 {q}")
        messages.append({"role": "user", "content": q})
        try:
            #r = agent.invoke({"messages": messages})
            r = invoke_with_fallback(messages)
            messages = r["messages"]
            last = messages[-1]
            a = _extract_last_text(r)
            print(f"✅ {wrap_text(a, width=92)}")
        except Exception as e:
            print(f"❌ {e}")
    print("\n✅ Тестування завершено.")

run_automatic_tests()



🧪 ТЕСТУВАННЯ StudyMate

Тест #1
------------------------------------------------------------
👤 Привіт! Що ти вмієш?
✅ Привіт! Я — освітній асистент **StudyMate**, і я допомагаю учням і студентам із точними
науками (математикою, фізикою та хімією).

Ось чим я можу бути корисним:
1. **Пошук і пояснення формул** — знайду потрібну формулу з математики, фізики чи хімії та
детально поясню, що означає кожна змінна.
2. **Розбір задач (у тому числі за фото)** — допоможу оформити «Дано» та «Знайти», перевести
величини у систему СІ та підібрати формулу для розв'язання.
3. **Конвертація фізичних величин** — переведу швидкість, температуру, енергію, тиск тощо з
одних одиниць в інші (наприклад, км/год у м/с).
4. **Довідка про хімічні елементи** — підкажу атомну масу, електронну конфігурацію та
основні властивості будь-якого елемента з таблиці Менделєєва.
5. **Планування підготовки до іспитів** — допоможу розрахувати, скільки тем варто вивчати
щодня та чи вистачить тобі часу на підготовку.

З чого п